# Labo 40 · Carnet Colab — Un vrai serveur MCP

Le serveur du service de la formation de TechNord, écrit avec la trousse Python officielle de MCP, puis un client qui s'y branche, puis un petit modèle de langage qui choisit lui-même l'outil à appeler.

**Avant de commencer** : *Exécution → Modifier le type d'exécution → GPU (T4)* (pour la partie 3 seulement), puis *Fichier → Enregistrer une copie dans Drive*.

À la fin, recopie tes résultats dans la case « Carnet Colab » de la page du labo.

In [ ]:
import subprocess, sys
# La trousse évolue vite : on fige une version majeure pour que le carnet reste stable.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp>=1.10,<2"], check=False)
import json

## 1. Le serveur

Le serveur est un programme à part : on l'écrit dans un fichier. Les **annotations de types** deviennent le schéma des paramètres, et la **chaîne de documentation** devient la description que lira le modèle.

**À toi** : complète l'outil `inscrire_liste_attente` (il lève une `ValueError` si la formation est inconnue ou si elle n'est pas complète) et la requête `accueil`.

In [ ]:
%%writefile serveur_formation.py
from mcp.server.fastmcp import FastMCP

HORAIRE = {"IA-101": "lundi et mercredi, 13 h à 16 h, salle Atlas", "WEB-210": "mardi, 8 h à 11 h, salle Polaris",
           "SEC-150": "jeudi, 10 h à 12 h, salle Orion"}
PLACES = {"IA-101": 3, "WEB-210": 0, "SEC-150": 12}
ATTENTE = {c: [] for c in PLACES}
REGLEMENT = ["Article 1. Présence. L'employé inscrit assiste à toutes les séances de sa formation.",
             "Article 2. Absences. Au-delà de 10 % d'absences non motivées, l'accès à l'évaluation finale peut être refusé.",
             "Article 3. Absence à une évaluation. Une pièce justificative doit être fournie dans les 5 jours ouvrables.",
             "Article 4. Retards. Un travail remis en retard perd 10 % de sa valeur par jour, jusqu'à 5 jours."]

serveur = FastMCP("service-formation")

@serveur.tool()
def horaire(formation: str) -> str:
    """Horaire d'une formation ce trimestre (jours, heures, salle). formation : code de la formation, par exemple IA-101."""
    if formation not in HORAIRE:
        raise ValueError(f"formation inconnue : {formation}")
    return HORAIRE[formation]

@serveur.tool()
def places_disponibles(formation: str) -> int:
    """Nombre de places libres dans une formation; 0 signifie que la formation est complète."""
    if formation not in PLACES:
        raise ValueError(f"formation inconnue : {formation}")
    return PLACES[formation]

# À toi : l'outil inscrire_liste_attente(formation: str, employe: str) -> str

@serveur.resource("reglement://article/{n}")
def article(n: int) -> str:
    """Le texte d'un article du règlement de formation."""
    return REGLEMENT[int(n) - 1]

# À toi : la requête accueil(equipe: str) -> str

if __name__ == "__main__":
    serveur.run()          # transport stdio

## 2. Le client

Le client lance le serveur comme un sous-programme et lui parle par stdio. Dans un carnet, on peut utiliser `await` directement dans une cellule.

**À toi** : liste les outils (nom et description), appelle `places_disponibles` pour une formation valide et une formation inexistante (regarde `isError`), lis la ressource `reglement://article/4` et obtiens la requête `accueil`.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

parametres = StdioServerParameters(command=sys.executable, args=["serveur_formation.py"])
JOURNAL_SERVEUR = open("serveur.log", "w")     # les messages du serveur (dans un carnet, la sortie d'erreur ne convient pas)

async def demo():
    async with stdio_client(parametres, errlog=JOURNAL_SERVEUR) as (lecture, ecriture):
        async with ClientSession(lecture, ecriture) as session:
            await session.initialize()
            # À toi
            pass

await demo()

## 3. Un modèle qui choisit l'outil

Le petit modèle du cours 38 n'a pas d'appel d'outils intégré : on lui donne la liste des outils dans le message système et on lui demande de répondre **seulement** en JSON, `{"outil": ..., "arguments": {...}}`. Le client exécute l'appel par MCP et renvoie le résultat au modèle, qui rédige la réponse. C'est la boucle de l'appel d'outils, faite à la main.

**À toi** : complète `repondre(question)` et essaie-la avec deux questions (une sur l'horaire, une sur les places). Le modèle choisit-il le bon outil? Des arguments valides?

In [ ]:
import torch, re
from transformers import AutoTokenizer, AutoModelForCausalLM
nom = "Qwen/Qwen2.5-0.5B-Instruct"
decoupeur = AutoTokenizer.from_pretrained(nom)
modele = AutoModelForCausalLM.from_pretrained(nom, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
modele = modele.to("cuda" if torch.cuda.is_available() else "cpu")

def discuter(messages, max_jetons=150):
    texte = decoupeur.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    entree = decoupeur(texte, return_tensors="pt").to(modele.device)
    sortie = modele.generate(**entree, max_new_tokens=max_jetons, do_sample=False)
    return decoupeur.decode(sortie[0][entree["input_ids"].shape[1]:], skip_special_tokens=True)

async def repondre(question):
    # À toi : lister les outils, demander le choix au modèle (JSON), appeler l'outil, faire rédiger la réponse
    pass

## À remettre

Recopie dans la case « Carnet Colab » de la page du labo : la liste des outils obtenue par le client, le résultat des deux appels à `places_disponibles` (dont celui en erreur), le texte de l'article 4, et, pour deux questions, l'outil et les arguments choisis par le modèle, puis sa réponse finale.